In [ ]:
import pandas as pd
import json
import ast
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

In [4]:
## Load 
paper_nodes = pd.read_csv("../../outputs/paper_nodes.csv")
oa_df = pd.read_csv("../../outputs/intermediate/openalex_metadata_full.csv")

print("Total papers:", len(paper_nodes))
print("OpenAlex papers:", len(oa_df))

# Build OpenAlex ID → global ID map

oa_id_map = dict(
    zip(oa_df["openalex_id"], oa_df["global_paper_id"])
)

print("OpenAlex ID map size:", len(oa_id_map))

Total papers: 2628
OpenAlex papers: 2529
OpenAlex ID map size: 2436


In [ ]:
# Build citation edges (safer parsing and logging)
edges = []
parsing_errors = 0
parsing_examples = []
missing_years_skipped = 0

for idx, row in oa_df.iterrows():

    source_global = row.get("global_paper_id")
    source_year = row.get("year")
    referenced = row.get("referenced_works")

    if pd.isna(referenced):
        continue

    # referenced_works stored as a stringified list in various formats → try json, then ast.literal_eval
    referenced_parsed = referenced
    if isinstance(referenced, str):
        try:
            referenced_parsed = json.loads(referenced)
        except Exception:
            try:
                referenced_parsed = ast.literal_eval(referenced)
            except Exception:
                parsing_errors += 1
                if len(parsing_examples) < 10:
                    parsing_examples.append(str(referenced)[:200])
                continue

    # Ensure we have an iterable of referenced OA ids
    if not isinstance(referenced_parsed, (list, tuple, set)):
        continue

    for ref_oa_id in referenced_parsed:

        # Skip if source year missing (can't place citation temporally)
        if pd.isna(source_year):
            missing_years_skipped += 1
            continue

        # Keep only citations within our paper universe
        if ref_oa_id in oa_id_map:

            target_global = oa_id_map[ref_oa_id]

            # Drop self-citations
            if source_global == target_global:
                continue

            edges.append({
                "source": source_global,
                "target": target_global,
                "year": source_year
            })

# Logging summary of parsing issues
logging.info(f"Citation parsing errors (rows skipped): {parsing_errors}")
if parsing_examples:
    logging.info("Sample problematic `referenced_works` values:")
    for ex in parsing_examples:
        logging.info(f"  {ex}")
if missing_years_skipped:
    logging.info(f"Skipped {missing_years_skipped} references due to missing source year")

In [6]:
# Create DataFrame
citation_edges = pd.DataFrame(edges)

print("Raw citation edges:", len(citation_edges))

# Remove duplicates
citation_edges = citation_edges.drop_duplicates()

print("After deduplication:", len(citation_edges))

Raw citation edges: 7425
After deduplication: 7425


In [10]:
# Final Sanity Checks
print("\nSANITY CHECKS")

print("Duplicate edges:",
      citation_edges.duplicated().sum())

print("Missing year values:",
      citation_edges["year"].isnull().sum())

assert citation_edges["source"].isin(
    paper_nodes["node_id"]
).all()

assert citation_edges["target"].isin(
    paper_nodes["node_id"]
).all()


SANITY CHECKS
Duplicate edges: 0
Missing year values: 0


In [8]:
# Save
citation_edges.to_csv("../../outputs/citation_edges.csv", index=False)

print("\nSaved: citation_edges.csv")


Saved: citation_edges.csv
